In [2]:
import sys
sys.path.insert(0,'/mnt/AEA8F340A8F3059D/sportsbet/ai-engine')

In [3]:
from fastapi import APIRouter,HTTPException,Request
from agents.supervisor import getAgent
from agents.state import AgentState
from tools.mongodbtools import fetchLiveMatches,fetchMatch
from config.constants import INTENT_SMART_ALERT
from services.mongodb import getDb
from config.constants import COLL_AI_INSIGHT
from models.schemas import validateAlertTriggerRequest
import logging

In [4]:
router=APIRouter()
logger=logging.getLogger(__name__)

In [5]:
@router.get("/alerts")
async def getAlerts(userId:str="",limit:int=20):
    db=getDb()
    insights=db[COLL_AI_INSIGHT].find(
        {
            "type":"smart-alert"
        }
    ).sort("createdAt",-1).limit(limit)
    alerts=[]
    async for a in insights:
        a["_id"]=str(a["_id"])
        alerts.append(a)
    return{
        "success":True,
        "data":{
            "alerts":alerts
        },
        "error":""
    }

In [9]:
@router.post("/alerts/trigger")
async def triggerAlert(request:Request):
    body=await request.json()
    params=validateAlertTriggerRequest(body)
    match=await fetchMatch(params["matchId"])
    if not match:
        raise HTTPException(status_code=404,detail="Match not found")
    agent=getAgent()
    if agent is None:
        raise HTTPException(status_code=500,detail="Agent not ready")
    try:
        state=AgentState(
            query="Generate smart alert for the match",
            intent=INTENT_SMART_ALERT,
            confidence=1.0,
            slots={
                "matchId":params["matchId"]
            },
            context={
                "matchId":params["matchId"],
                "pageType":"match"
            },
            user_id="system",
            match_data=match,
            live_score=match.get("liveScore",{})
        )
        result=await agent.ainvoke(state)
        return{
            "success":True,
            "data":result.get("output",{}),
            "error":""
        }
    except Exception as e:
        logger.error(f"Alert failed {str(e)}")
        raise HTTPException(status_code=500,detail=str(e))